In [ ]:
# !pip install ultralytics


In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from PIL import Image

In [2]:
# Set paths & parameters
data_dir = '../dataset_split'
results_dir = '../results'
model_dir = '../models'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

classes = ['biological', 'plastic']
img_size = 224
epochs = 30
batch_size = 16
model_name = 'yolov8n-cls.pt'

In [3]:
# Check dataset structure
for split in ['train', 'val', 'test']:
    for cls in classes:
        folder = os.path.join(data_dir, split, cls)
        count = len([
            f for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        print(f'{split}/{cls}: {count} images')

train/biological: 729 images
train/plastic: 729 images
val/biological: 156 images
val/plastic: 156 images
test/biological: 157 images
test/plastic: 157 images


In [4]:
# Load & Train YOLOv8
model = YOLO(model_name)

results = model.train(
    data = data_dir,
    epochs = epochs,
    imgsz = img_size,
    batch = batch_size,
    # name = 'yolov8_smartbin',
    project = 'runs',
    name = 'yolov8_smartbin',
    patience = 5,
    verbose = True
)

New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.40  Python-3.13.5 torch-2.10.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../dataset_split, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_s

In [5]:
import glob

found = glob.glob('runs/**/best.pt',  recursive=True) + \
        glob.glob('../**/best.pt', recursive=True)

# Keep only yolov8 results
found = [f for f in found if 'yolov8' in f]

print('Found best.pt files:')
for f in found:
    print(f'  → {f}')

if found:
    best_model_path = max(found, key=os.path.getmtime)
    best_model      = YOLO(best_model_path)
    print(f'\nBest model loaded from:\n   {best_model_path}')
else:
    print('best.pt not found — run Cell 5 first!')

Found best.pt files:
  → runs\classify\runs\yolov8_smartbin\weights\best.pt
  → runs\classify\runs\yolov8_smartbin-2\weights\best.pt
  → runs\classify\runs\yolov8_smartbin-3\weights\best.pt
  → ..\notebooks\runs\classify\runs\yolov8_smartbin\weights\best.pt
  → ..\notebooks\runs\classify\runs\yolov8_smartbin-2\weights\best.pt
  → ..\notebooks\runs\classify\runs\yolov8_smartbin-3\weights\best.pt

Best model loaded from:
   runs\classify\runs\yolov8_smartbin-3\weights\best.pt


In [6]:
# Evaluate on Test Set

# Reset as plain lists first (important!)
y_true = []
y_pred = []

for label_idx, cls in enumerate(classes):
    cls_dir = os.path.join(data_dir, 'test', cls)
    images  = glob.glob(os.path.join(cls_dir, '*.jpg'))  + \
              glob.glob(os.path.join(cls_dir, '*.jpeg')) + \
              glob.glob(os.path.join(cls_dir, '*.png'))

    print(f'Predicting {cls}: {len(images)} images...')
    for img_path in images:
        result   = best_model.predict(img_path, verbose=False)
        pred_idx = result[0].probs.top1
        y_true.append(label_idx)
        y_pred.append(pred_idx)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = accuracy_score(y_true, y_pred)
print(f'\nTest Accuracy : {acc*100:.2f}%')
print(f'   Total images  : {len(y_true)}')
print(f'   Correct       : {np.sum(y_true == y_pred)}')
print(f'   Wrong         : {np.sum(y_true != y_pred)}')

Predicting biological: 157 images...
Predicting plastic: 157 images...

Test Accuracy : 94.59%
   Total images  : 314
   Correct       : 297
   Wrong         : 17


In [7]:
# Classification report 
print('CLASSIFICATION REPORT YOLOv8')
print('='*60)
print(classification_report(
    y_true, y_pred,
    target_names = classes
))

CLASSIFICATION REPORT YOLOv8
              precision    recall  f1-score   support

  biological       0.91      0.99      0.95       157
     plastic       0.99      0.90      0.94       157

    accuracy                           0.95       314
   macro avg       0.95      0.95      0.95       314
weighted avg       0.95      0.95      0.95       314



In [8]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot       = True,
    fmt         = 'd',
    cmap        = 'Blues',
    xticklabels = classes,
    yticklabels = classes,
    linewidths  = 0.5
)
ax.set_title('Confusion Matrix — YOLOv8', fontsize=13, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'yolov8_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/yolov8_confusion_matrix.png')
print(f'\n  True Biological  (correct): {cm[0][0]}')
print(f'  False Plastic    (wrong)  : {cm[0][1]}')
print(f'  False Biological (wrong)  : {cm[1][0]}')
print(f'  True Plastic     (correct): {cm[1][1]}')

<Figure size 600x500 with 2 Axes>

Saved to results/yolov8_confusion_matrix.png

  True Biological  (correct): 156
  False Plastic    (wrong)  : 1
  False Biological (wrong)  : 16
  True Plastic     (correct): 141


In [9]:
# Sample predictions
all_paths = []
for cls in classes:
    cls_dir = os.path.join(data_dir, 'test', cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_paths.append((os.path.join(cls_dir, f), cls))

random.seed(42)
samples = random.sample(all_paths, min(8, len(all_paths)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Predictions — YOLOv8',
             fontsize=13, fontweight='bold')
axes = axes.flatten()

for i, (path, true_label) in enumerate(samples):
    result     = best_model.predict(path, verbose=False)
    pred_idx   = result[0].probs.top1
    pred_label = classes[pred_idx]
    confidence = result[0].probs.top1conf.item() * 100
    is_correct = pred_label == true_label

    img = Image.open(path).convert('RGB')
    axes[i].imshow(img)
    axes[i].axis('off')
    color  = '#4CAF50' if is_correct else '#F44336'
    status = '✓' if is_correct else '✗'
    axes[i].set_title(
        f'{status} Pred: {pred_label}\n{confidence:.1f}% | True: {true_label}',
        color=color, fontsize=8, fontweight='bold'
    )

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'yolov8_sample_predictions.png'),
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1600x800 with 8 Axes>

In [10]:
# Final summary
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)

print('='*50)
print('      EVALUATION SUMMARY — YOLOv8')
print('='*50)
print(f'  Test Accuracy : {acc*100:.2f}%')
print(f'  Precision     : {precision*100:.2f}%')
print(f'  Recall        : {recall*100:.2f}%')
print(f'  F1-Score      : {f1*100:.2f}%')
print('='*50)

      EVALUATION SUMMARY — YOLOv8
  Test Accuracy : 94.59%
  Precision     : 99.30%
  Recall        : 89.81%
  F1-Score      : 94.31%
